# 1d spatiotemporal bilateral filtering

In [ ]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import numpy as np

folder = Path.cwd()
if not (folder / 'filters.py').exists():
    folder = folder / '1d_signal'
sys.path.insert(0, str(folder))

from filters import cross_bilateral_filter, laplacian_smoothing_filter
from optimizers import adam_optimizer, spatiotemporal_optimizer

rng = np.random.default_rng(4)

In [ ]:
def make_piecewise_smooth_target(size):
    x = np.linspace(0, 1, size)
    target = np.zeros(size)
    target[(x >= 0.10) & (x < 0.32)] = 0.35
    target[(x >= 0.32) & (x < 0.55)] = 0.35 + 0.4 * (x[(x >= 0.32) & (x < 0.55)] - 0.32) / 0.23
    target[(x >= 0.55) & (x < 0.78)] = -0.25
    target[x >= 0.78] = 0.55
    return target


def run_optimizer(optimizer, theta_start, gradient_fn, iterations, snapshots):
    theta = theta_start.copy()
    history = {0: theta.copy()}

    for iteration in range(1, iterations + 1):
        gradient = gradient_fn(theta, iteration - 1)
        theta = optimizer.step(theta, gradient)

        if iteration in snapshots:
            history[iteration] = theta.copy()

    return history


def run_gradient_descent(theta_start, gradient_fn, learning_rate, iterations, snapshots):
    theta = theta_start.copy()
    history = {0: theta.copy()}

    for iteration in range(1, iterations + 1):
        theta = theta - learning_rate * gradient_fn(theta, iteration - 1)

        if iteration in snapshots:
            history[iteration] = theta.copy()

    return history


def plot_histories(histories, target, snapshots, title):
    names = list(histories)
    figure, axes = plt.subplots(len(names), len(snapshots), figsize=(3.2 * len(snapshots), 2.2 * len(names)), sharex=True, sharey=True)
    axes = np.atleast_2d(axes)

    for row, name in enumerate(names):
        for column, iteration in enumerate(snapshots):
            axis = axes[row, column]
            axis.plot(target, color='black', linewidth=1.5, label='reference')
            axis.plot(histories[name][iteration], color='tab:blue', linewidth=1.2, label=name)
            axis.set_title(f'{name}, iteration {iteration}')

    axes[0, 0].legend(loc='upper left')
    figure.suptitle(title, y=1.02)
    figure.tight_layout()
    return figure

## figure 2, anisotropic objective


In [ ]:
size = 256
iterations = 500
snapshots = [0, 5, 50, 250, 500]

target = make_piecewise_smooth_target(size)
theta_start = np.zeros(size)
a = rng.normal(size=(size, size))
ata = a.T @ a

def anisotropic_gradient(theta, iteration):
    '''gradient of (theta - target)^t A^t A (theta - target).'''
    return 2 * ata @ (theta - target)

laplacian = laplacian_smoothing_filter(strength=5.0)
bilateral = cross_bilateral_filter(sigma_spatial=5.0, sigma_data=0.08)

histories_figure_2 = {
    'gradient descent': run_gradient_descent(theta_start, anisotropic_gradient, 0.00015, iterations, snapshots),
    'adam': run_optimizer(adam_optimizer(lr=0.02, b_1=0.2, b_2=0.2), theta_start, anisotropic_gradient, iterations, snapshots),
    'laplacian': run_optimizer(spatiotemporal_optimizer(lr=0.05, filter=laplacian, b_1=0.2, b_2=0.2), theta_start, anisotropic_gradient, iterations, snapshots),
    'cross bilateral': run_optimizer(spatiotemporal_optimizer(lr=0.05, filter=bilateral, b_1=0.2, b_2=0.2), theta_start, anisotropic_gradient, iterations, snapshots),
}

plot_histories(histories_figure_2, target, snapshots, 'figure 2 style: anisotropic objective');

## figure 3 style: isotropic objective

f(theta) = ||theta - target||^2 

In [ ]:
size = 256
iterations = 500
snapshots = [0, 5, 50, 250, 500]
noise_std = 0.35

target = make_piecewise_smooth_target(size)
theta_start = np.zeros(size)
noise = rng.normal(scale=noise_std, size=(iterations, size))

def noisy_isotropic_gradient(theta, iteration):
    '''gradient of squared error plus zero-mean gaussian noise.'''
    return 2 * (theta - target) + noise[iteration]

laplacian = laplacian_smoothing_filter(strength=5.0)
bilateral = cross_bilateral_filter(sigma_spatial=5.0, sigma_data=0.08)

histories_figure_3 = {
    'gradient descent': run_gradient_descent(theta_start, noisy_isotropic_gradient, 0.10, iterations, snapshots),
    'adam': run_optimizer(adam_optimizer(lr=0.05, b_1=0.2, b_2=0.2), theta_start, noisy_isotropic_gradient, iterations, snapshots),
    'laplacian': run_optimizer(spatiotemporal_optimizer(lr=0.10, filter=laplacian, b_1=0.2, b_2=0.2), theta_start, noisy_isotropic_gradient, iterations, snapshots),
    'cross bilateral': run_optimizer(spatiotemporal_optimizer(lr=0.10, filter=bilateral, b_1=0.2, b_2=0.2), theta_start, noisy_isotropic_gradient, iterations, snapshots),
}

plot_histories(histories_figure_3, target, snapshots, 'figure 3 style: isotropic objective with noisy gradients');